# COCO → QLoRA → RePOPE 实验流水线

本 notebook 只编排 `scripts/` 中经过注释的可复用代码。E1/E2/E3 各自形成训练 → dev 选择 → RePOPE test 的独立闭环；RePOPE 结果只做阶段测试，不能反向参与任何 checkpoint 选择。

## 0. 实验契约

固定 4-bit NF4、BF16、256 visual tokens、prompt、答案解析和 effective batch=16。E1/E2/E3/E4 每次仅改变预注册变量；E1/E2/E3 均在 dev 冻结唯一 adapter 后立即运行 RePOPE test，test 结果不参与选择。

In [ ]:
from pathlib import Path
import json, os, re, subprocess, sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
PY = Path(sys.executable)
def run(*args):
    # Stream stdout/stderr into the notebook so download bars and the real traceback stay visible.
    child_env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONIOENCODING': 'utf-8'}
    process = subprocess.Popen([str(PY), *map(str, args)], cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1, env=child_env)
    for char in iter(lambda: process.stdout.read(1), ''):
        print(char, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, process.args)
    return code

DEV_BASELINE = ROOT/'results/qlora_evaluations/e0_base_2k_dev_256vt_metrics.json'
def select_dev_adapter(experiment, manual_checkpoint=None):
    """Run the pre-registered dev-only rule and resolve exactly one adapter for test."""
    evaluation_dir = ROOT/'results/qlora_evaluations'
    candidates = sorted(
        evaluation_dir.glob(f'{experiment}_checkpoint-*_dev_256vt_metrics.json'),
        key=lambda path: int(re.search(r'checkpoint-(\d+)', path.name).group(1)),
    )
    if not candidates:
        raise FileNotFoundError(f'No dev checkpoint metrics found for {experiment}. Run its dev evaluation cell first.')
    selection_path = ROOT/f'results/qlora_runs/{experiment}/selection_dev2k.json'
    run('scripts/select_checkpoint.py', '--baseline', DEV_BASELINE, '--candidates', *candidates, '--output', selection_path)
    selection = json.loads(selection_path.read_text(encoding='utf-8'))
    if selection['selected'] is not None:
        match = re.search(r'checkpoint-(\d+)', Path(selection['selected']['path']).name)
        checkpoint_name = f'checkpoint-{match.group(1)}'
        decision = 'pre_registered'
    elif manual_checkpoint is not None:
        checkpoint_name = f'checkpoint-{int(manual_checkpoint)}'
        decision = 'manual_override_after_no_eligible_checkpoint'
    else:
        raise RuntimeError(
            f'{experiment}: no eligible checkpoint. Set {experiment.upper()}_MANUAL_CHECKPOINT explicitly before test; '
            'do not use RePOPE results to make that choice.'
        )
    adapter = ROOT/f'results/qlora_runs/{experiment}'/checkpoint_name
    if not adapter.is_dir():
        raise FileNotFoundError(adapter)
    print({'experiment': experiment, 'adapter': str(adapter), 'decision': decision, 'selection': str(selection_path)})
    return adapter

def run_repope_test(experiment, adapter, max_visual_tokens=256):
    """Run one frozen RePOPE test after dev selection; never feed this result back into selection."""
    run_name = f'{experiment}_{adapter.name}'
    metrics_path = ROOT/f'results/qlora_evaluations/{run_name}_repope_{max_visual_tokens}vt_metrics.json'
    if metrics_path.exists():
        print(f'Skip existing RePOPE result: {metrics_path}')
    else:
        run('scripts/evaluate_qlora.py', '--dataset', 'repope', '--adapter', adapter, '--run-name', run_name, '--max-visual-tokens', str(max_visual_tokens))
    return json.loads(metrics_path.read_text(encoding='utf-8'))

json.loads((ROOT/'configs/qlora_experiments.json').read_text())

## 1. 环境审计

确认 CUDA、Transformers、PEFT、bitsandbytes 和现有 RePOPE 文件后再启动任何 GPU 作业。

In [ ]:
run('-m', 'pip', 'show', 'transformers', 'peft', 'bitsandbytes', 'torch', 'datasets')
subprocess.run(['nvidia-smi'], check=True)

## 2. 下载 COCO 与构造无泄露 train/dev

本单元下载官方 annotations 和最终被采样到的图片；使用精确 10,000 张训练图片构造 16k E1 问题，E3 逐行复用相同图片并仅改变负样本比例，另构造独立的 3k dev，并写入 image-id 防泄露 manifest。

In [ ]:
# 首次执行会联网下载；若已存在数据可去掉 --download。
run('scripts/prepare_coco_repope_style.py','--download')

## 3. 数据配额与防泄露检查

应看到 E1=8k yes/8k no，dev 的三个 split 各为 500 yes/500 no，且 train/dev/RePOPE image_id 零交集。

In [ ]:
manifest = json.loads((ROOT/'data/processed/coco_repope_style_manifest.json').read_text())
manifest

## 4. 工程测速：阶段 1

C-off 与 C-on 的 micro-batch=1、accumulation=16，唯一变量为 checkpointing。优先运行 C-off 检查 OOM；每组仅 5 optimizer steps，前 1 step 是 warm-up。每个配置在独立 Python/CUDA 子进程中运行，避免显存基线相互污染。

In [ ]:
run('scripts/benchmark_qlora_speed.py', '--phase', '1')

## 5. 工程测速：阶段 2 与配置锁定

固定阶段 1 选出的 checkpointing 设置，比较 B1/B2；B4 因显存压力过高而预注册跳过。每组在独立 Python/CUDA 子进程中运行，总共 16 个 optimizer steps，前 5 步 warm-up、后 11 步计入统计；脚本选择无 OOM 且 samples/s 最高的设置并写入锁定配置。

In [ ]:
run('scripts/benchmark_qlora_speed.py', '--phase', '2')
locked = json.loads((ROOT/'results/qlora_locked_training_config.json').read_text()); locked

## 6. Smoke test

固定 128 条数据，使用锁定的 micro-batch=2、accumulation=8、effective batch=16 跑 8 个 optimizer steps，恰好遍历一次。训练前审计 5 条 answer-only labels、全部可训练参数和 optimizer groups；动态检查有限 loss、非零梯度、LoRA 参数变化与显存稳定；结束时仅保存一次 adapter，并验证重载前后贪心推理一致。不运行 dev。

In [ ]:
run('scripts/run_qlora.py', '--experiment', 'smoke', '--output-dir', 'results/qlora_runs/smoke')

## 7. E0：冻结基座在 dev 上的参考

这是 checkpoint 约束的唯一基线；不使用 RePOPE。当前统一读取固定的 2,000 条分层 dev 子集（1,000 yes / 1,000 no）。本单元直接将已保存的 E0 3k 预测对齐到这 2k 并重算指标，不重新加载模型或推理。

In [ ]:
# 复用已有 3k E0 predictions，秒级生成同一 2k 子集上的新基线。
run('scripts/recompute_e0_dev2k.py')

## 8. E1：LLM attention-only QLoRA

视觉 encoder 与 merger 冻结；只训练 LLM q/k/v/o adapter。训练与 dev 已拆成两个独立单元：本节先只完成训练并保存 checkpoints，不会自动进入评测。

In [ ]:
run('scripts/run_qlora.py', '--experiment', 'e1')

### E1 checkpoints：手动启动 2k dev 评测

确认训练已结束后再运行下一单元。它会依次评测 checkpoint-500/1000/1500/2000；必须显式传入 `--start`，避免训练结束后误触发。

In [ ]:
run('scripts/evaluate_checkpoints.py', '--run-dir', 'results/qlora_runs/e1', '--run-name', 'e1', '--dataset', 'dev', '--start')

### E1：dev 选择后立即运行 RePOPE test

本单元先用固定 2k dev 和预注册规则生成 `selection_dev2k.json`，再冻结唯一 adapter 运行一次 RePOPE 256-token test。当前 E1 严格规则为 `no_eligible_checkpoint`，因此显式记录人工候选 checkpoint-1000；RePOPE 结果不得用于反向修改该选择。

In [ ]:
E1_MANUAL_CHECKPOINT = 1000  # 严格规则无合格项；这是 test 前冻结的探索性人工候选
E1_TEST_ADAPTER = select_dev_adapter('e1', manual_checkpoint=E1_MANUAL_CHECKPOINT)
E1_REPOPE = run_repope_test('e1', E1_TEST_ADAPTER, max_visual_tokens=256)
E1_REPOPE['metrics']['overall']

## 9. E2：Merger LoRA

在 E1 的全部设置上，只新增 `model.visual.merger.mlp.0/2` 的 LoRA。它不是从头训练 merger，因此 adapter LR 与 LLM LoRA 一致，均为 1e-4。

In [ ]:
run('scripts/run_qlora.py', '--experiment', 'e2')

### E2 checkpoints：手动启动 2k dev 评测

In [ ]:
run('scripts/evaluate_checkpoints.py', '--run-dir', 'results/qlora_runs/e2', '--run-name', 'e2', '--dataset', 'dev', '--start')
# 将 E1/E2 的 *_checkpoint_metrics.json 中的 metrics 路径传给 select_checkpoint.py。

### E2：dev 选择后立即运行 RePOPE test

先执行 dev-only 预注册规则，再对冻结的唯一 adapter 运行 RePOPE 256-token test。如果规则返回 `no_eligible_checkpoint`，本单元会停止；必须在查看 dev 指标后显式填写 `E2_MANUAL_CHECKPOINT`，不能根据 RePOPE 结果挑 checkpoint。

In [ ]:
E2_MANUAL_CHECKPOINT = None  # 仅当 dev 规则无合格项时，在 test 前显式填 500/1000/1500/2000
E2_TEST_ADAPTER = select_dev_adapter('e2', manual_checkpoint=1000)
E2_REPOPE = run_repope_test('e2', E2_TEST_ADAPTER, max_visual_tokens=256)
E2_REPOPE['metrics']['overall']

## 10. E3：adversarial 负样本加权

将 E1/E2 中 dev 选出的架构填到 `--parent-architecture`；从基础模型重新训练，只把负样本比例改为 12.5/12.5/25。

In [ ]:
PARENT = 'e1'  # 仅在 selection_e1_e2.json 显示 selected 后替换
run('scripts/run_qlora.py', '--experiment', 'e3', '--parent-architecture', PARENT)

### E3 checkpoints：手动启动 2k dev 评测

E3 训练结束后先独立评测全部 checkpoints。该单元补齐原 notebook 缺失的 E3 dev 选择步骤。

In [ ]:
run('scripts/evaluate_checkpoints.py', '--run-dir', 'results/qlora_runs/e3', '--run-name', 'e3', '--dataset', 'dev', '--start')

### E3：正式 RePOPE test（formal selected checkpoint）

E3 的 2k dev 预注册规则唯一选中 `checkpoint-1000`：Precision=98.10%，adversarial FPR=3.90%，两项均满足约束。因此它是唯一的正式 E3 checkpoint，并且只对这个 adapter 运行正式 RePOPE 256-token test。RePOPE 结果不能反向改变这项选择。

In [ ]:
# Formal analysis: the pre-registered dev rule must resolve checkpoint-1000.
E3_FORMAL_ADAPTER = select_dev_adapter('e3')
assert E3_FORMAL_ADAPTER.name == 'checkpoint-1000', E3_FORMAL_ADAPTER
E3_FORMAL_REPOPE = run_repope_test('e3', E3_FORMAL_ADAPTER, max_visual_tokens=256)
E3_FORMAL_REPOPE['metrics']['overall']

### E3：探索性 RePOPE 对照（不参与正式选择）

`checkpoint-2000` 在不施加 Precision/FPR 约束时具有最高 Accuracy/F1（95.95%/95.93%）、更高 Recall（95.50%）且 Yes Ratio 更接近 50%，但不满足严格约束。因此它预先固定为 exploratory best-unconstrained/high-recall checkpoint。下面的 RePOPE 结果只用于观察 sensitivity–hallucination trade-off；无论结果如何，都不能替换正式的 checkpoint-1000。

In [ ]:
# Exploratory analysis only: this cell never calls select_dev_adapter and cannot alter the formal choice.
E3_EXPLORATORY_CHECKPOINT = 2000
E3_EXPLORATORY_ADAPTER = ROOT/'results/qlora_runs/e3'/f'checkpoint-{E3_EXPLORATORY_CHECKPOINT}'
assert E3_EXPLORATORY_ADAPTER.is_dir(), E3_EXPLORATORY_ADAPTER
E3_EXPLORATORY_REPOPE = run_repope_test('e3_exploratory_high_recall', E3_EXPLORATORY_ADAPTER, max_visual_tokens=256)

# Keep the two roles explicit in the saved notebook output and later report.
def repope_tradeoff_summary(role, checkpoint, payload):
    overall = payload['metrics']['overall']
    adversarial = payload['metrics']['adversarial']
    return {
        'role': role,
        'checkpoint': checkpoint,
        'accuracy': overall['accuracy'],
        'precision': overall['precision'],
        'recall': overall['recall'],
        'f1': overall['f1'],
        'yes_ratio': overall['yes_ratio'],
        'overall_fpr': overall['false_positive_rate'],
        'adversarial_fpr': adversarial['false_positive_rate'],
    }

E3_REPOPE_ROLE_COMPARISON = [
    repope_tradeoff_summary('formal_selected', 1000, E3_FORMAL_REPOPE),
    repope_tradeoff_summary('exploratory_best_unconstrained_high_recall', E3_EXPLORATORY_CHECKPOINT, E3_EXPLORATORY_REPOPE),
]
E3_REPOPE_ROLE_COMPARISON

## 11. E4：LLM all-linear LoRA

E4 从基础 4-bit 模型重新训练，使用与 E3 相同的 adversarial 加权数据。E1 是此前选定的基础架构，因此视觉 encoder 与 merger 均冻结；唯一实验变量是将 LLM LoRA 从 attention-only 的 q/k/v/o 扩展到 q/k/v/o/gate/up/down。训练、dev 选择与 RePOPE test 在下方严格拆开。

In [ ]:
# 阶段 A：只训练 E4；此单元不会评测 dev 或 RePOPE。
E4_PARENT_ARCHITECTURE = 'e1'  # E1/E2 架构比较后固定为 attention-only base architecture
run('scripts/run_qlora.py', '--experiment', 'e4', '--parent-architecture', E4_PARENT_ARCHITECTURE)

### E4 阶段 B：手动启动 2k dev checkpoint 评测

只在 E4 训练完成后运行。它依次评测 checkpoint-500/1000/1500/2000，输出 Accuracy、Precision、Recall、F1、overall FPR、adversarial FPR 与 Yes Ratio。此阶段只使用固定 2k COCO dev，不接触 RePOPE。

In [ ]:
run('scripts/evaluate_checkpoints.py', '--run-dir', 'results/qlora_runs/e4', '--run-name', 'e4', '--dataset', 'dev', '--start')

### E4 阶段 C：冻结 dev-selected adapter 后运行正式 RePOPE

E4 的 dev 结果中，checkpoint-1000 是唯一满足严格约束的 checkpoint（Adversarial FPR ≤ 4.6036%，Overall Precision ≥ 96.7074%），因此预注册的正式选择固定为它。该单元只对 checkpoint-1000 运行一次 RePOPE 256-token test；不得用 RePOPE 结果重新选择 checkpoint。

In [ ]:
# 阶段 C：formal test。checkpoint-1000 是 dev 上唯一的 strict-eligible 候选。
E4_FORMAL_ADAPTER = select_dev_adapter('e4')
assert E4_FORMAL_ADAPTER.name == 'checkpoint-1000', E4_FORMAL_ADAPTER
E4_FORMAL_REPOPE = run_repope_test('e4', E4_FORMAL_ADAPTER, max_visual_tokens=256)
E4_FORMAL_REPOPE['metrics']['overall']

### E4 探索性分析：checkpoint-2000（不改变正式选择）

checkpoint-2000 的 dev F1（95.57%）和 Recall（95.00%）明显高于正式 checkpoint-1000 的 Recall（90.60%），但 Precision=96.15%、Adversarial FPR=6.61%，未满足严格约束。因此它只作为预先明确标注的 exploratory / high-recall 对照：可单独跑一次 RePOPE 来观察 sensitivity–hallucination trade-off，但任何 RePOPE 数值均不能反过来改写 formal selected checkpoint-1000。

In [ ]:
# Exploratory analysis only: this cell never calls select_dev_adapter and cannot alter the formal choice.
E4_EXPLORATORY_CHECKPOINT = 2000
E4_EXPLORATORY_ADAPTER = ROOT/'results/qlora_runs/e4'/f'checkpoint-{E4_EXPLORATORY_CHECKPOINT}'
assert E4_EXPLORATORY_ADAPTER.is_dir(), E4_EXPLORATORY_ADAPTER
E4_EXPLORATORY_REPOPE = run_repope_test('e4_exploratory_high_recall', E4_EXPLORATORY_ADAPTER, max_visual_tokens=256)

# Keep the formal and exploratory roles explicit in the notebook output and final report.
def e4_repope_tradeoff_summary(role, checkpoint, payload):
    overall = payload['metrics']['overall']
    adversarial = payload['metrics']['adversarial']
    return {
        'role': role,
        'checkpoint': checkpoint,
        'accuracy': overall['accuracy'],
        'precision': overall['precision'],
        'recall': overall['recall'],
        'f1': overall['f1'],
        'yes_ratio': overall['yes_ratio'],
        'adversarial_fpr': adversarial['false_positive_rate'],
    }

E4_REPOPE_ROLE_COMPARISON = [
    e4_repope_tradeoff_summary('formal_selected', 1000, E4_FORMAL_REPOPE),
    e4_repope_tradeoff_summary('exploratory_best_unconstrained_high_recall', E4_EXPLORATORY_CHECKPOINT, E4_EXPLORATORY_REPOPE),
]
E4_REPOPE_ROLE_COMPARISON

## 12. E-best-512：同一正式 adapter 的 256/512 visual-token 消融

目的：先在 E1–E4 间仅依据 **dev** 的预注册规则确定全局 formal winner，再检验该已冻结 adapter 的更高视觉 token 上限是否有益。输入是各实验的 `selection_dev2k.json`、全局 winner 已完成的 RePOPE 256-token 结果；输出是同一 adapter 的 RePOPE 512-token 结果和逐项差值。唯一变化是推理时 `max_visual_tokens: 256 → 512`，adapter 权重、提示词、答案解析和 RePOPE 样本均不变。

运行前提：E1–E4 的 dev checkpoint 评测均已完成，并且全局 winner 已完成它自己的正式阶段 C。该阶段产生的 256-token metrics 是配对基线；本节会拒绝在缺少它时直接跑 512，避免把正式 test 和 512 消融混为一谈。当前 dev 审计结果中 E1/E2 无 strict-eligible checkpoint，E3-1000 与 E4-1000 合格；按“Recall 优先”的预注册规则，E3-1000（93.10%）胜过 E4-1000（90.60%）。E3/E4 的 checkpoint-2000 都是探索性对照，不能参与本节的 formal winner 选择。

In [ ]:
# 1) 跨 E1–E4 读取 dev-only selection；绝不读取任何 RePOPE metric 来选模型。
def global_formal_dev_selection(experiments=('e1', 'e2', 'e3', 'e4'), recall_tie=0.002):
    candidates, excluded = [], {}
    for experiment in experiments:
        path = ROOT/f'results/qlora_runs/{experiment}/selection_dev2k.json'
        assert path.exists(), f'Missing dev selection: {path}'
        selection = json.loads(path.read_text(encoding='utf-8'))
        selected = selection.get('selected')
        if selected is None:
            excluded[experiment] = 'no_strict_eligible_checkpoint'
            continue
        metric_path = Path(selected['path'])
        checkpoint = int(re.search(r'checkpoint-(\d+)', metric_path.name).group(1))
        candidates.append({
            'experiment': experiment, 'checkpoint': checkpoint,
            'recall': selected['recall'], 'f1': selected['f1'],
            'adversarial_fpr': selected['adversarial_fpr'],
            'yes_ratio_distance': abs(selected['yes_ratio'] - selected['true_yes_ratio']),
        })
    assert candidates, 'No strict-eligible experiment-level formal candidate exists.'
    # Pre-registered cross-experiment rule: highest Recall; only within 0.2 pp use F1, lower adv-FPR, then Yes Ratio.
    top_recall = max(row['recall'] for row in candidates)
    near_recall = [row for row in candidates if top_recall - row['recall'] < recall_tie]
    winner = sorted(near_recall, key=lambda row: (-row['f1'], row['adversarial_fpr'], row['yes_ratio_distance']))[0]
    return winner, candidates, excluded

FINAL_DEV_SELECTION, FINAL_DEV_CANDIDATES, FINAL_DEV_EXCLUDED = global_formal_dev_selection()
assert (FINAL_DEV_SELECTION['experiment'], FINAL_DEV_SELECTION['checkpoint']) == ('e3', 1000), FINAL_DEV_SELECTION
FINAL_FORMAL_EXPERIMENT = FINAL_DEV_SELECTION['experiment']
FINAL_FORMAL_CHECKPOINT = FINAL_DEV_SELECTION['checkpoint']
BEST_ADAPTER = ROOT/f'results/qlora_runs/{FINAL_FORMAL_EXPERIMENT}'/f'checkpoint-{FINAL_FORMAL_CHECKPOINT}'
assert BEST_ADAPTER.is_dir(), BEST_ADAPTER

# 256-token 必须已经由该实验的阶段 C 跑完；这里仅加载，不会重复或补跑正式 test。
FORMAL_256_METRICS_PATH = ROOT/'results/qlora_evaluations'/f'{FINAL_FORMAL_EXPERIMENT}_{BEST_ADAPTER.name}_repope_256vt_metrics.json'
assert FORMAL_256_METRICS_PATH.exists(), (
    f'Missing formal 256-token result: {FORMAL_256_METRICS_PATH}. Run E4 phase C first.'
)
BEST_REPOPE_256 = json.loads(FORMAL_256_METRICS_PATH.read_text(encoding='utf-8'))

# 唯一新执行的评测：完全相同的 adapter 与 RePOPE，只把视觉 token 上限改为 512。
BEST_REPOPE_512 = run_repope_test(FINAL_FORMAL_EXPERIMENT, BEST_ADAPTER, max_visual_tokens=512)

# 成对汇总 overall 与三个 split；delta = 512-token - 256-token。
def paired_256_512_summary(metrics_256, metrics_512):
    rows = []
    for split in ('overall', 'random', 'popular', 'adversarial'):
        before = metrics_256['metrics'][split]
        after = metrics_512['metrics'][split]
        row = {'split': split}
        for key in ('accuracy', 'precision', 'recall', 'f1', 'false_positive_rate', 'yes_ratio'):
            if key in before and key in after:
                row[f'{key}_256'] = before[key]
                row[f'{key}_512'] = after[key]
                row[f'{key}_delta_512_minus_256'] = after[key] - before[key]
        rows.append(row)
    return rows

BEST_256_512_PAIRED_COMPARISON = paired_256_512_summary(BEST_REPOPE_256, BEST_REPOPE_512)
BEST_256_512_PAIRED_COMPARISON

## 13. Fresh E0：冻结 base model 的 RePOPE 256/512 对照

早期 `results/qwen2vl2b_baseline_repope_metrics.json` 是将既有 POPE prediction 与 RePOPE 标签重新对齐后的重算结果，**没有重新运行模型推理**。本节补做 fresh E0：重新加载未挂载任何 LoRA adapter 的 4-bit Qwen2-VL-2B base model，使用当前 `evaluate_qlora.py` 的 prompt、贪心生成、答案解析和官方 RePOPE 对齐流程，对全部 8,185 条 RePOPE 保留样本重新生成预测。

这不是 checkpoint 或架构选择步骤，且不会改变 E1–E4 的 dev selection。它的目的只是给 E3-1000 的 256/512 结果提供同协议、可公平比较的原始模型基线。256 与 512 必须串行运行，避免同时加载两个模型实例。

In [ ]:
# Fresh E0 通用运行器：故意不接受 adapter，确保评测的是冻结的原始 base model。
E0_FRESH_RUN_NAME = 'e0_fresh_base'
E0_FRESH_EVALUATION_DIR = ROOT/'results/qlora_evaluations'

def run_fresh_e0_once(max_visual_tokens):
    assert max_visual_tokens in (256, 512)
    common_args = (
        'scripts/evaluate_qlora.py', '--dataset', 'repope',
        '--run-name', E0_FRESH_RUN_NAME,
    )
    # The absence of --adapter is the experimental contract for fresh E0.
    assert '--adapter' not in common_args
    prefix = f'{E0_FRESH_RUN_NAME}_repope_{max_visual_tokens}vt'
    metrics_path = E0_FRESH_EVALUATION_DIR/f'{prefix}_metrics.json'
    predictions_path = E0_FRESH_EVALUATION_DIR/f'{prefix}_predictions.jsonl'
    if metrics_path.exists() and predictions_path.exists():
        print(f'Skip completed Fresh E0 {max_visual_tokens}vt: {metrics_path}')
    elif metrics_path.exists() != predictions_path.exists():
        raise RuntimeError(f'Incomplete Fresh E0 output; inspect before rerun: {metrics_path}, {predictions_path}')
    else:
        run(*common_args, '--max-visual-tokens', str(max_visual_tokens))
    assert metrics_path.exists() and predictions_path.exists()
    payload = json.loads(metrics_path.read_text(encoding='utf-8'))
    assert payload['dataset'] == 'repope'
    assert payload['adapter'] is None, payload['adapter']
    assert payload['max_visual_tokens'] == max_visual_tokens
    assert payload['dataset_sample_count'] == 8185
    assert all(payload['metrics'][split]['unknown'] == 0 for split in ('random', 'popular', 'adversarial', 'overall'))
    return payload


### Fresh E0 · 256 visual tokens

运行纯 base model 的正式 256-token RePOPE 基线。运行完成后会保存 metrics 和逐题 predictions；若两个文件已经完整存在，则安全跳过。

In [ ]:
E0_FRESH_REPOPE_256 = run_fresh_e0_once(256)
E0_FRESH_REPOPE_256['metrics']['overall']

### Fresh E0 · 512 visual tokens

在 256 完成并释放模型后，再顺序运行同一冻结 base model 的 512-token RePOPE。唯一实验变量是 `max_visual_tokens`。

In [ ]:
E0_FRESH_REPOPE_512 = run_fresh_e0_once(512)
E0_FRESH_REPOPE_512['metrics']['overall']

### Fresh E0 与 E3-1000 的成对汇总

本单元只读取已有 JSON，不启动任何推理。它并列展示 Fresh E0 与 formal selected E3-1000 在 256 / 512 下的 overall 及三个 RePOPE split 指标，并单独给出 Fresh E0 的 `512 - 256` 差值。该表仅用于报告，不参与 dev-based 的模型选择。

In [ ]:
# 只读汇总：若 kernel 已重启，从固定产物重新加载四份 metrics。
def load_repope_metrics(run_name, max_visual_tokens):
    path = ROOT/'results/qlora_evaluations'/f'{run_name}_repope_{max_visual_tokens}vt_metrics.json'
    assert path.exists(), f'Missing metrics: {path}'
    return json.loads(path.read_text(encoding='utf-8'))

E0_FRESH_REPOPE_256 = load_repope_metrics('e0_fresh_base', 256)
E0_FRESH_REPOPE_512 = load_repope_metrics('e0_fresh_base', 512)
E3_FORMAL_REPOPE_256 = load_repope_metrics('e3_checkpoint-1000', 256)
E3_FORMAL_REPOPE_512 = load_repope_metrics('e3_checkpoint-1000', 512)

for payload, tokens in ((E0_FRESH_REPOPE_256, 256), (E0_FRESH_REPOPE_512, 512)):
    assert payload['adapter'] is None
    assert payload['dataset_sample_count'] == 8185 and payload['max_visual_tokens'] == tokens
    assert all(payload['metrics'][split]['unknown'] == 0 for split in ('random', 'popular', 'adversarial', 'overall'))

def report_row(model, tokens, split, payload):
    metric = payload['metrics'][split]
    return {
        'model': model, 'visual_tokens': tokens, 'split': split,
        **{key: metric[key] for key in ('accuracy', 'precision', 'recall', 'f1', 'false_positive_rate', 'yes_ratio')},
    }

REPOPE_E0_FRESH_VS_E3 = [
    report_row(model, tokens, split, payload)
    for model, tokens, payload in (
        ('E0-fresh-base', 256, E0_FRESH_REPOPE_256),
        ('E0-fresh-base', 512, E0_FRESH_REPOPE_512),
        ('E3-checkpoint-1000', 256, E3_FORMAL_REPOPE_256),
        ('E3-checkpoint-1000', 512, E3_FORMAL_REPOPE_512),
    )
    for split in ('overall', 'random', 'popular', 'adversarial')
]

E0_FRESH_512_MINUS_256 = []
for split in ('overall', 'random', 'popular', 'adversarial'):
    before = E0_FRESH_REPOPE_256['metrics'][split]
    after = E0_FRESH_REPOPE_512['metrics'][split]
    E0_FRESH_512_MINUS_256.append({
        'model': 'E0-fresh-base', 'split': split,
        **{f'{key}_delta_512_minus_256': after[key] - before[key]
           for key in ('accuracy', 'precision', 'recall', 'f1', 'false_positive_rate', 'yes_ratio')},
    })

{'fresh_e0_vs_e3': REPOPE_E0_FRESH_VS_E3, 'fresh_e0_512_minus_256': E0_FRESH_512_MINUS_256}

## 14. 复现记录

最终报告需保存：数据 manifest、锁定测速配置、run manifest、dev selection JSON、RePOPE 256/512 metrics 与逐题 prediction JSONL。

In [ ]:
# ============================================================
# E0 old-vs-fresh inference consistency audit
# 直接接在 fresh E0 RePOPE 256vt cell 后面运行
# 不重新推理，只审计已有 predictions
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import json
import csv

PROJECT_ROOT = ROOT
RESULTS_DIR = PROJECT_ROOT / "results"

OLD_E0_PRED_PATH = RESULTS_DIR / "qwen2vl2b_baseline_predictions.jsonl"
FRESH_E0_PRED_PATH = RESULTS_DIR / "qlora_evaluations" / "e0_fresh_base_repope_256vt_predictions.jsonl"

AUDIT_DIR = RESULTS_DIR / "e0_inference_consistency_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

FLIP_JSONL = AUDIT_DIR / "e0_old_vs_fresh_flips.jsonl"
FLIP_UNIQUE_JSONL = AUDIT_DIR / "e0_old_vs_fresh_unique_flips.jsonl"
FLIP_CSV = AUDIT_DIR / "e0_old_vs_fresh_flips.csv"


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


old_rows = read_jsonl(OLD_E0_PRED_PATH)
fresh_rows = read_jsonl(FRESH_E0_PRED_PATH)

print("OLD rows  :", len(old_rows))
print("FRESH rows:", len(fresh_rows))

assert len(old_rows) == 9000, len(old_rows)
assert len(fresh_rows) == 8185, len(fresh_rows)


# ------------------------------------------------------------
# 1. 用 split + question_id 对齐
# ------------------------------------------------------------

def make_key(row):
    return (
        str(row["split"]).lower(),
        int(row["question_id"])
    )


old_map = {make_key(x): x for x in old_rows}
fresh_map = {make_key(x): x for x in fresh_rows}

missing_in_old = [
    key for key in fresh_map
    if key not in old_map
]

assert not missing_in_old, (
    f"{len(missing_in_old)} fresh samples cannot be found in old predictions."
)

print("Aligned fresh samples:", len(fresh_map))


# ------------------------------------------------------------
# 2. 检查 image / question 是否真的相同
# ------------------------------------------------------------

image_mismatch = []
question_mismatch = []
prediction_flips = []

for key, fresh in fresh_map.items():
    old = old_map[key]

    old_image = str(old.get("image_source", "")).strip()
    fresh_image = str(fresh.get("image_source", "")).strip()

    old_question = str(old.get("question", ""))
    fresh_question = str(fresh.get("question", ""))

    if old_image != fresh_image:
        image_mismatch.append({
            "key": key,
            "old_image": old_image,
            "fresh_image": fresh_image,
        })

    if old_question != fresh_question:
        question_mismatch.append({
            "key": key,
            "old_question": old_question,
            "fresh_question": fresh_question,
        })

    old_pred = str(old.get("prediction", "")).strip().lower()
    fresh_pred = str(fresh.get("prediction", "")).strip().lower()

    if old_pred != fresh_pred:
        label = str(
            fresh.get(
                "label",
                fresh.get("ground_truth", "")
            )
        ).strip().lower()

        prediction_flips.append({
            "split": fresh["split"],
            "question_id": int(fresh["question_id"]),
            "image_source": fresh["image_source"],
            "question": fresh["question"],
            "repope_label": label,
            "old_prediction": old_pred,
            "fresh_prediction": fresh_pred,
            "old_raw_answer": old.get("raw_answer"),
            "fresh_raw_answer": fresh.get("raw_answer"),
            "old_elapsed_seconds": old.get("elapsed_seconds"),
            "fresh_elapsed_seconds": fresh.get("elapsed_seconds"),
        })


print()
print("=" * 70)
print("INPUT CONSISTENCY")
print("=" * 70)
print("image mismatch   :", len(image_mismatch))
print("question mismatch:", len(question_mismatch))
print("prediction flips :", len(prediction_flips))

assert len(image_mismatch) == 0, (
    "Old/fresh image_source is not identical."
)

assert len(question_mismatch) == 0, (
    "Old/fresh question text is not identical."
)

print("\n=> 对齐后的 8185 条样本具有完全相同的 image_source + question。")


# ------------------------------------------------------------
# 3. 翻转类型统计
# ------------------------------------------------------------

flip_counter = Counter()

for row in prediction_flips:
    transition = (
        row["repope_label"],
        row["old_prediction"],
        row["fresh_prediction"],
    )
    flip_counter[transition] += 1

print()
print("=" * 70)
print("PREDICTION FLIP TYPES")
print("=" * 70)

for (label, old_pred, fresh_pred), n in sorted(flip_counter.items()):
    print(
        f"GT={label:3s} | "
        f"{old_pred:3s} -> {fresh_pred:3s} | "
        f"{n:3d}"
    )


# ------------------------------------------------------------
# 4. 统计每个 split 的变化
# ------------------------------------------------------------

split_counter = Counter()

for row in prediction_flips:
    split_counter[row["split"]] += 1

print()
print("=" * 70)
print("FLIPS BY SPLIT")
print("=" * 70)

for split, n in split_counter.items():
    print(f"{split:12s}: {n}")


# ------------------------------------------------------------
# 5. 唯一 image-question pair
#    因为同一个 positive question 可能同时出现在三个 POPE split
# ------------------------------------------------------------

pair_groups = defaultdict(list)

for row in prediction_flips:
    pair_key = (
        row["image_source"],
        row["question"],
    )
    pair_groups[pair_key].append(row)

print()
print("=" * 70)
print("UNIQUE FLIP CASES")
print("=" * 70)
print("prediction-level flips :", len(prediction_flips))
print("unique image-question  :", len(pair_groups))

repeat_distribution = Counter(
    len(rows)
    for rows in pair_groups.values()
)

print("repeat distribution    :", dict(sorted(repeat_distribution.items())))


unique_flip_rows = []

for unique_id, ((image_source, question), rows) in enumerate(
    pair_groups.items(),
    start=1
):
    representative = rows[0]

    # 同一个 image-question 在不同 split 中
    # old/fresh 预测理论上应该一致
    old_predictions = sorted(
        set(x["old_prediction"] for x in rows)
    )
    fresh_predictions = sorted(
        set(x["fresh_prediction"] for x in rows)
    )

    unique_flip_rows.append({
        "unique_id": unique_id,
        "image_source": image_source,
        "question": question,
        "repope_label": representative["repope_label"],
        "old_prediction": representative["old_prediction"],
        "fresh_prediction": representative["fresh_prediction"],
        "appears_in_splits": [
            x["split"] for x in rows
        ],
        "question_ids": [
            x["question_id"] for x in rows
        ],
        "repeat_count": len(rows),
        "old_predictions_across_repeats": old_predictions,
        "fresh_predictions_across_repeats": fresh_predictions,
    })


# ------------------------------------------------------------
# 6. 检查 old / fresh 各自内部是否 deterministic
# ------------------------------------------------------------

old_pair_preds = defaultdict(set)
fresh_pair_preds = defaultdict(set)

for key, fresh in fresh_map.items():
    old = old_map[key]

    pair = (
        fresh["image_source"],
        fresh["question"],
    )

    old_pair_preds[pair].add(
        str(old["prediction"]).lower()
    )

    fresh_pair_preds[pair].add(
        str(fresh["prediction"]).lower()
    )

old_internal_inconsistent = {
    k: v
    for k, v in old_pair_preds.items()
    if len(v) > 1
}

fresh_internal_inconsistent = {
    k: v
    for k, v in fresh_pair_preds.items()
    if len(v) > 1
}

print()
print("=" * 70)
print("WITHIN-RUN DETERMINISM")
print("=" * 70)
print(
    "old repeated-pair inconsistencies  :",
    len(old_internal_inconsistent)
)
print(
    "fresh repeated-pair inconsistencies:",
    len(fresh_internal_inconsistent)
)

if not old_internal_inconsistent and not fresh_internal_inconsistent:
    print(
        "\n=> 同一个 image-question 在各自 run 内都给出稳定答案；"
        "\n   这不像 sampling 随机波动，更像两次推理流程存在系统性细微差异。"
    )


# ------------------------------------------------------------
# 7. 保存完整 flip audit
# ------------------------------------------------------------

with open(FLIP_JSONL, "w", encoding="utf-8") as f:
    for row in prediction_flips:
        f.write(
            json.dumps(row, ensure_ascii=False)
            + "\n"
        )

with open(FLIP_UNIQUE_JSONL, "w", encoding="utf-8") as f:
    for row in unique_flip_rows:
        f.write(
            json.dumps(row, ensure_ascii=False)
            + "\n"
        )

csv_fields = [
    "split",
    "question_id",
    "image_source",
    "question",
    "repope_label",
    "old_prediction",
    "fresh_prediction",
    "old_raw_answer",
    "fresh_raw_answer",
    "old_elapsed_seconds",
    "fresh_elapsed_seconds",
]

with open(
    FLIP_CSV,
    "w",
    encoding="utf-8-sig",
    newline=""
) as f:
    writer = csv.DictWriter(
        f,
        fieldnames=csv_fields
    )
    writer.writeheader()

    for row in prediction_flips:
        writer.writerow({
            key: row.get(key)
            for key in csv_fields
        })


# ------------------------------------------------------------
# 8. 推理速度对比
# ------------------------------------------------------------

def mean_elapsed(rows):
    vals = [
        float(x["elapsed_seconds"])
        for x in rows
        if x.get("elapsed_seconds") is not None
    ]

    return sum(vals) / len(vals)


print()
print("=" * 70)
print("INFERENCE SPEED")
print("=" * 70)

for split in ["random", "popular", "adversarial"]:
    old_split = [
        old_map[key]
        for key in fresh_map
        if key[0] == split
    ]

    fresh_split = [
        fresh_map[key]
        for key in fresh_map
        if key[0] == split
    ]

    old_mean = mean_elapsed(old_split)
    fresh_mean = mean_elapsed(fresh_split)

    speedup = (
        old_mean / fresh_mean
        if fresh_mean > 0
        else float("nan")
    )

    print(
        f"{split:12s} | "
        f"old={old_mean:.4f}s | "
        f"fresh={fresh_mean:.4f}s | "
        f"old/fresh={speedup:.3f}x"
    )


# ------------------------------------------------------------
# 9. 打印所有 unique flip cases
# ------------------------------------------------------------

print()
print("=" * 70)
print("ALL UNIQUE FLIP CASES")
print("=" * 70)

for row in unique_flip_rows:
    print(
        f"\n[{row['unique_id']:02d}] "
        f"{row['old_prediction'].upper()} -> "
        f"{row['fresh_prediction'].upper()} | "
        f"GT={row['repope_label'].upper()} | "
        f"splits={row['appears_in_splits']}"
    )
    print("image   :", row["image_source"])
    print("question:", row["question"])


print()
print("=" * 70)
print("SAVED")
print("=" * 70)
print("all flips  :", FLIP_JSONL)
print("unique     :", FLIP_UNIQUE_JSONL)
print("csv        :", FLIP_CSV)

In [ ]:
# ============================================================
# Audit current run_repope_test implementation / environment
# ============================================================

import inspect
import platform
import sys
import torch
import transformers

print("=" * 70)
print("CURRENT PYTHON / TORCH / TRANSFORMERS ENVIRONMENT")
print("=" * 70)

print("Python       :", sys.version.replace("\n", " "))
print("Platform     :", platform.platform())
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("CUDA runtime :", torch.version.cuda)
print(
    "CUDA device  :",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

try:
    import bitsandbytes as bnb
    print(
        "bitsandbytes:",
        getattr(bnb, "__version__", "unknown")
    )
except Exception as e:
    print("bitsandbytes: unavailable:", repr(e))


print()
print("=" * 70)
print("run_repope_test SIGNATURE")
print("=" * 70)

print(inspect.signature(run_repope_test))


print()
print("=" * 70)
print("run_repope_test SOURCE")
print("=" * 70)

try:
    RUN_REPOPE_TEST_SOURCE = inspect.getsource(
        run_repope_test
    )

    print(RUN_REPOPE_TEST_SOURCE)

except Exception as e:
    RUN_REPOPE_TEST_SOURCE = None
    print(
        "inspect.getsource(run_repope_test) failed:",
        repr(e)
    )


# 保存下来，防止 notebook 后面改代码以后不知道当前版本是什么
AUDIT_DIR = ROOT / "results" / "e0_inference_consistency_audit"

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if RUN_REPOPE_TEST_SOURCE is not None:
    source_path = (
        AUDIT_DIR
        / "current_run_repope_test_source.py"
    )

    source_path.write_text(
        RUN_REPOPE_TEST_SOURCE,
        encoding="utf-8"
    )

    print(
        "\nSaved current run_repope_test source:",
        source_path
    )

In [ ]:
# ============================================================
# 1. 确认当前 fresh E0 的基础加载配置
# ============================================================

# 显式导入，保证 kernel 重启后也可单独执行本单元。
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.qlora_common import (
    PIXELS_PER_VISUAL_TOKEN,
    config as load_qlora_config,
)

cfg = load_qlora_config()

print("model_path              :", cfg["model_path"])
print("PIXELS_PER_VISUAL_TOKEN :", PIXELS_PER_VISUAL_TOKEN)
print("256 max_pixels          :", 256 * PIXELS_PER_VISUAL_TOKEN)
print("512 max_pixels          :", 512 * PIXELS_PER_VISUAL_TOKEN)

import torch

print("bf16 supported          :", torch.cuda.is_bf16_supported())
print(
    "current compute dtype   :",
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

In [ ]:
# ============================================================
# 2. 找旧 E0 baseline 当时到底是用哪段代码跑的
#    搜 .py + .ipynb
# ============================================================

from pathlib import Path
import json
import os

SEARCH_TERMS = [
    "qwen2vl2b_baseline_predictions",
    "lmms-lab/POPE",
    "Qwen2VLForConditionalGeneration",
]

print("=" * 80)
print("SEARCHING OLD E0 IMPLEMENTATION")
print("=" * 80)

# ---------- Python files ----------
for path in ROOT.rglob("*.py"):
    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    matched = [
        term for term in SEARCH_TERMS
        if term in text
    ]

    if matched:
        print()
        print("=" * 80)
        print("PYTHON FILE:", path)
        print("MATCHED    :", matched)
        print("=" * 80)

        lines = text.splitlines()

        for i, line in enumerate(lines, start=1):
            if any(term in line for term in SEARCH_TERMS):
                start = max(1, i - 40)
                end = min(len(lines), i + 100)

                for lineno in range(start, end + 1):
                    print(
                        f"{lineno:4d} | "
                        f"{lines[lineno - 1]}"
                    )

                print()


# ---------- Jupyter notebooks ----------
for path in ROOT.rglob("*.ipynb"):
    try:
        notebook = json.loads(
            path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )
    except Exception:
        continue

    for cell_idx, cell in enumerate(
        notebook.get("cells", [])
    ):
        source = "".join(
            cell.get("source", [])
        )

        matched = [
            term for term in SEARCH_TERMS
            if term in source
        ]

        if matched:
            print()
            print("=" * 80)
            print("NOTEBOOK:", path)
            print("CELL    :", cell_idx)
            print("MATCHED :", matched)
            print("=" * 80)
            print(source)

In [ ]:
# ============================================================
# E0 forensic audit - Cell 1
# Load the 22 unique old-vs-fresh flip cases
# ============================================================

from pathlib import Path
import json
import os

from datasets import load_dataset, DownloadConfig

AUDIT_DIR = ROOT / "results" / "e0_inference_consistency_audit"
FLIP_UNIQUE_JSONL = AUDIT_DIR / "e0_old_vs_fresh_unique_flips.jsonl"

CACHE_DIR = Path(os.environ.get("HF_DATASETS_CACHE", Path.home() / ".cache" / "huggingface" / "datasets"))

assert FLIP_UNIQUE_JSONL.exists(), FLIP_UNIQUE_JSONL

unique_flip_rows = []

with FLIP_UNIQUE_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            unique_flip_rows.append(json.loads(line))

print("unique flip cases:", len(unique_flip_rows))

assert len(unique_flip_rows) == 22, (
    f"Expected 22 unique flips, got {len(unique_flip_rows)}"
)


# ------------------------------------------------------------
# Load the same cached POPE dataset used by the old E0
# ------------------------------------------------------------

pope = load_dataset(
    "lmms-lab/POPE",
    "Full",
    cache_dir=str(CACHE_DIR),
    download_config=DownloadConfig(local_files_only=True),
)

qid_to_index = {
    split: {
        str(sample["question_id"]): i
        for i, sample in enumerate(pope[split])
    }
    for split in ("random", "popular", "adversarial")
}


# ------------------------------------------------------------
# Resolve every unique flip back to a real POPE image
# ------------------------------------------------------------

flip_cases = []

for row in unique_flip_rows:

    # 每个 unique image-question 只取一个实际 occurrence 即可
    split = str(row["appears_in_splits"][0])
    question_id = str(row["question_ids"][0])

    dataset_index = qid_to_index[split][question_id]
    sample = pope[split][dataset_index]

    # 验证还是我们之前对齐的同一个样本
    assert Path(str(sample["image_source"])).stem == Path(
        str(row["image_source"])
    ).stem

    assert str(sample["question"]) == str(row["question"])

    flip_cases.append({
        "unique_id": row["unique_id"],
        "split": split,
        "question_id": question_id,
        "dataset_index": dataset_index,

        "image_source": sample["image_source"],
        "image": sample["image"],
        "question": str(sample["question"]),

        "old_prediction": str(row["old_prediction"]).lower(),
        "fresh_prediction": str(row["fresh_prediction"]).lower(),
        "repope_label": str(row["repope_label"]).lower(),
    })


print("resolved cases:", len(flip_cases))

print()
print("First case:")
print("image       :", flip_cases[0]["image_source"])
print("question    :", flip_cases[0]["question"])
print("old         :", flip_cases[0]["old_prediction"])
print("fresh       :", flip_cases[0]["fresh_prediction"])
print("image mode  :", flip_cases[0]["image"].mode)

In [ ]:
# ============================================================
# E0 forensic audit - Cell 2
# Define loaders + exact old/current inference pipelines
# ============================================================

import gc
from collections import Counter

import torch
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2VLForConditionalGeneration,
)

from scripts.qlora_common import (
    config,
    PIXELS_PER_VISUAL_TOKEN,
    load_quantized_model,
)

from scripts.evaluate_pope import normalize_answer


MAX_VISUAL_TOKENS = 256


# ------------------------------------------------------------
# Generic base-model loader
# ------------------------------------------------------------

def load_e0_variant(
    attn_implementation=None,
    prepare_kbit=False,
):
    """
    Load the base Qwen2-VL model without LoRA.

    attn_implementation:
        None     -> old-style unspecified backend
        "eager"  -> force eager
        "sdpa"   -> force SDPA

    prepare_kbit:
        False    -> old-style inference model
        True     -> apply PEFT prepare_model_for_kbit_training
    """

    from peft import prepare_model_for_kbit_training

    cfg = config()

    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )

    processor = AutoProcessor.from_pretrained(
        cfg["model_path"],
        local_files_only=True,
        max_pixels=(
            MAX_VISUAL_TOKENS
            * PIXELS_PER_VISUAL_TOKEN
        ),
    )

    model_kwargs = dict(
        local_files_only=True,
        quantization_config=quant,
        torch_dtype=dtype,
        device_map="auto",
    )

    if attn_implementation is not None:
        model_kwargs["attn_implementation"] = (
            attn_implementation
        )

    model = Qwen2VLForConditionalGeneration.from_pretrained(
        cfg["model_path"],
        **model_kwargs,
    )

    if prepare_kbit:

        # 完全照当前 load_quantized_model()
        model.config.use_cache = False

        if hasattr(model.config, "text_config"):
            model.config.text_config.use_cache = False

        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=False,
        )

    model.eval()

    return model, processor, dtype


# ------------------------------------------------------------
# OLD inference pipeline
# scripts/evaluate_pope.py
# ------------------------------------------------------------

def infer_e0_legacy(
    model,
    processor,
    case,
):
    image = case["image"].convert("RGB")

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": (
                        f'{case["question"]}\n'
                        'Answer using only "yes" or "no".'
                    ),
                },
            ],
        }
    ]

    prompt = processor.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True,
    )

    device = (
        model
        .get_input_embeddings()
        .weight
        .device
    )

    inputs = processor(
        text=[prompt],
        images=[image],
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():

        ids = model.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    generated = ids[
        :,
        inputs["input_ids"].shape[1]:
    ]

    raw = processor.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return normalize_answer(raw), raw


# ------------------------------------------------------------
# CURRENT inference pipeline
# scripts/evaluate_qlora.py
# ------------------------------------------------------------

def infer_e0_current(
    model,
    processor,
    case,
):
    image = case["image"]

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": (
                        str(case["question"])
                        + '\nAnswer using only "yes" or "no".'
                    ),
                },
            ],
        }
    ]

    prompt = processor.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True,
    )

    device = (
        model
        .get_input_embeddings()
        .weight
        .device
    )

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():

        ids = model.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    raw = processor.batch_decode(
        ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )[0]

    return normalize_answer(raw), raw


# ------------------------------------------------------------
# Inspect actual model state
# ------------------------------------------------------------

def describe_model(model):

    attn_impl = getattr(
        model.config,
        "_attn_implementation",
        None,
    )

    dtype_counts = Counter(
        str(param.dtype)
        for param in model.parameters()
    )

    norm_dtype_counts = Counter(
        str(param.dtype)
        for name, param in model.named_parameters()
        if "norm" in name.lower()
    )

    return {
        "attn_implementation": attn_impl,
        "parameter_dtypes": dict(dtype_counts),
        "norm_dtypes": dict(norm_dtype_counts),
        "use_cache_top": getattr(
            model.config,
            "use_cache",
            None,
        ),
        "use_cache_text": getattr(
            getattr(model.config, "text_config", None),
            "use_cache",
            None,
        ),
    }


def clear_model(model=None):

    if model is not None:
        del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ============================================================
# E0 forensic audit - Cell 3
# Run controlled A/B/C/D comparison on the 22 flip cases
# ============================================================

audit_results = []
model_states = {}


def evaluate_loaded_model(
    variant_name,
    model,
    processor,
    pipeline,
):

    print()
    print("=" * 80)
    print(
        variant_name,
        "| pipeline =",
        pipeline,
    )
    print("=" * 80)

    state = describe_model(model)
    model_states[
        f"{variant_name}__{pipeline}"
    ] = state

    print("model state:")
    print(json.dumps(
        state,
        indent=2,
        default=str,
    ))

    infer_fn = (
        infer_e0_legacy
        if pipeline == "legacy"
        else infer_e0_current
    )

    rows = []

    for case in tqdm(
        flip_cases,
        desc=f"{variant_name}/{pipeline}",
    ):

        prediction, raw = infer_fn(
            model,
            processor,
            case,
        )

        row = {
            "variant": variant_name,
            "pipeline": pipeline,

            "unique_id": case["unique_id"],
            "split": case["split"],
            "question_id": case["question_id"],
            "image_source": case["image_source"],
            "question": case["question"],

            "old_prediction": case["old_prediction"],
            "fresh_prediction": case["fresh_prediction"],

            "prediction": prediction,
            "raw_answer": raw,

            "matches_old": (
                prediction
                == case["old_prediction"]
            ),

            "matches_fresh": (
                prediction
                == case["fresh_prediction"]
            ),
        }

        rows.append(row)
        audit_results.append(row)

    old_matches = sum(
        x["matches_old"]
        for x in rows
    )

    fresh_matches = sum(
        x["matches_fresh"]
        for x in rows
    )

    other = len(rows) - max(
        old_matches,
        fresh_matches,
    )

    print()
    print(
        f"matches OLD   : "
        f"{old_matches}/{len(rows)}"
    )

    print(
        f"matches FRESH : "
        f"{fresh_matches}/{len(rows)}"
    )

    print(
        f"other         : "
        f"{other}"
    )

    return rows


# ============================================================
# A. Old-style default attention, NO prepare_kbit
# ============================================================

print("\n\n### A: DEFAULT / NO KBIT PREP ###")

model, processor, dtype = load_e0_variant(
    attn_implementation=None,
    prepare_kbit=False,
)

# 先用真正的 legacy pipeline
evaluate_loaded_model(
    "A_default_no_kbit",
    model,
    processor,
    pipeline="legacy",
)

# 同一个模型再用 current pipeline
# 用来单独判断 padding/RGB/decode 等小差异是否有影响
evaluate_loaded_model(
    "A_default_no_kbit",
    model,
    processor,
    pipeline="current",
)

del model, processor
gc.collect()
torch.cuda.empty_cache()


# ============================================================
# B. Force EAGER, NO prepare_kbit
# ============================================================

print("\n\n### B: EAGER / NO KBIT PREP ###")

model, processor, dtype = load_e0_variant(
    attn_implementation="eager",
    prepare_kbit=False,
)

evaluate_loaded_model(
    "B_eager_no_kbit",
    model,
    processor,
    pipeline="current",
)

del model, processor
gc.collect()
torch.cuda.empty_cache()


# ============================================================
# C. Force SDPA, NO prepare_kbit
# ============================================================

print("\n\n### C: SDPA / NO KBIT PREP ###")

model, processor, dtype = load_e0_variant(
    attn_implementation="sdpa",
    prepare_kbit=False,
)

evaluate_loaded_model(
    "C_sdpa_no_kbit",
    model,
    processor,
    pipeline="current",
)

del model, processor
gc.collect()
torch.cuda.empty_cache()


# ============================================================
# D. CURRENT EXACT HELPER
# SDPA + prepare_model_for_kbit_training
# ============================================================

print("\n\n### D: CURRENT HELPER EXACT ###")

model, processor, dtype = load_quantized_model(
    "e1",
    False,
    MAX_VISUAL_TOKENS,
    attach_lora=False,
)

evaluate_loaded_model(
    "D_current_helper",
    model,
    processor,
    pipeline="current",
)

del model, processor
gc.collect()
torch.cuda.empty_cache()


# ============================================================
# Final summary
# ============================================================

print()
print()
print("=" * 90)
print("FINAL SUMMARY")
print("=" * 90)

groups = {}

for row in audit_results:

    key = (
        row["variant"],
        row["pipeline"],
    )

    groups.setdefault(
        key,
        []
    ).append(row)


for (variant, pipeline), rows in groups.items():

    old_matches = sum(
        row["matches_old"]
        for row in rows
    )

    fresh_matches = sum(
        row["matches_fresh"]
        for row in rows
    )

    print(
        f"{variant:25s} "
        f"{pipeline:8s} | "
        f"OLD={old_matches:2d}/22 | "
        f"FRESH={fresh_matches:2d}/22 | "
        f"attn="
        f"{model_states[f'{variant}__{pipeline}']['attn_implementation']}"
    )


# ============================================================
# Save audit
# ============================================================

RESULT_PATH = (
    AUDIT_DIR
    / "e0_loader_backend_forensic_audit.json"
)

RESULT_PATH.write_text(
    json.dumps(
        {
            "model_states": model_states,
            "results": audit_results,
        },
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print()
print("Saved:", RESULT_PATH)